# Human-in-the-Loop Workflow gamit ang Microsoft Agent Framework

## 🎯 Mga Layunin sa Pagkatuto

Sa notebook na ito, matututunan mo kung paano magpatupad ng **human-in-the-loop** workflows gamit ang `RequestInfoExecutor` ng Microsoft Agent Framework. Ang makapangyarihang pattern na ito ay nagpapahintulot sa iyo na pansamantalang ihinto ang mga AI workflow upang mangalap ng input mula sa tao, ginagawa ang iyong mga agent na interactive at binibigyan ang mga tao ng kontrol sa mahahalagang desisyon.

## 🔄 Ano ang Human-in-the-Loop?

**Human-in-the-loop (HITL)** ay isang disenyo ng pattern kung saan ang mga AI agent ay humihinto muna upang humingi ng input mula sa tao bago magpatuloy. Ito ay mahalaga para sa:

- ✅ **Mahahalagang desisyon** - Kumuha ng pag-apruba mula sa tao bago gumawa ng mahahalagang aksyon
- ✅ **Malabong sitwasyon** - Hayaan ang tao na linawin kapag nagdududa ang AI
- ✅ **Mga kagustuhan ng gumagamit** - Tanungin ang mga gumagamit upang pumili sa pagitan ng maraming opsyon
- ✅ **Pagsunod at kaligtasan** - Tiyakin ang pag-monitor ng tao para sa mga reguladong operasyon
- ✅ **Interaktibong karanasan** - Bumuo ng mga conversational agent na tumutugon sa input ng gumagamit

## 🏗️ Paano Ito Gumagana sa Microsoft Agent Framework

Nagbibigay ang framework ng tatlong pangunahing bahagi para sa HITL:

1. **`RequestInfoExecutor`** - Isang espesyal na executor na humihinto sa workflow at nagpapalabas ng `RequestInfoEvent`
2. **`RequestInfoMessage`** - Base class para sa mga typed request payload na ipinapadala sa mga tao
3. **`RequestResponse`** - Nag-uugnay sa mga tugon ng tao sa orihinal na mga kahilingan gamit ang `request_id`

**Pattern ng Workflow:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Ang Aming Halimbawa: Pag-book ng Hotel na may Kumpirmasyon mula sa Gumagamit

Magpapatuloy tayo sa conditional workflow sa pamamagitan ng pagdaragdag ng kumpirmasyon mula sa tao **bago** magmungkahi ng mga alternatibong destinasyon:

1. Humihiling ang gumagamit ng isang destinasyon (hal., "Paris")
2. Sinusuri ng `availability_agent` kung may mga bakanteng kuwarto
3. **Kung walang puwang** → tinatanong ng `confirmation_agent` "Gusto mo bang makita ang mga alternatibo?"
4. **Humihinto** ang workflow gamit ang `RequestInfoExecutor`
5. **Tumutugon ang tao** ng "oo" o "hindi" sa pamamagitan ng console input
6. Ang `decision_manager` ay nagro-route batay sa tugon:
   - **Oo** → Ipakita ang mga alternatibong destinasyon
   - **Hindi** → Kanselahin ang kahilingan sa booking
7. Ipakita ang huling resulta

Ipinapakita nito kung paano bigyan ang mga gumagamit ng kontrol sa mga mungkahi ng agent!

---

Simulan na natin! 🚀


## Hakbang 1: I-import ang Mga Kinakailangang Library

Ini-import natin ang mga karaniwang bahagi ng Agent Framework pati na rin ang **mga klaseng partikular sa human-in-the-loop**:
- `RequestInfoExecutor` - Executor na humihinto sa workflow para sa input ng tao
- `RequestInfoEvent` - Pangyayari na inilalabas kapag humihingi ng input mula sa tao
- `RequestInfoMessage` - Base na klase para sa mga typed na request payload
- `RequestResponse` - Nag-uugnay ng mga tugon ng tao sa mga request
- `WorkflowOutputEvent` - Pangyayari para sa pagtukoy ng mga output ng workflow


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Hakbang 2: Tukuyin ang mga Pydantic Models para sa mga Istrakturang Output

Tinutukoy ng mga modelong ito ang **schema** na ibabalik ng mga ahente. Pinananatili namin ang lahat ng mga modelo mula sa conditional workflow at nagdaragdag:

**Bago para sa Human-in-the-Loop:**
- `HumanFeedbackRequest` - Subclass ng `RequestInfoMessage` na tumutukoy sa request payload na ipinapadala sa mga tao
  - Naglalaman ng `prompt` (tanong na itatanong) at `destination` (konteksto tungkol sa hindi available na lungsod)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Hakbang 3: Gumawa ng Kagamitan para sa Pag-book ng Hotel

Parehong kagamitan mula sa conditional workflow - sinusuri kung may mga bakanteng kwarto sa destinasyon.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Hakbang 4: Tukuyin ang mga Kondisyon na Function para sa Routing

Kailangan natin ng **apat na kondisyon na function** para sa ating human-in-the-loop workflow:

**Mula sa conditional workflow:**
1. `has_availability_condition` - Nagro-route kapag MAY available na mga hotel
2. `no_availability_condition` - Nagro-route kapag WALANG available na mga hotel

**Bago para sa human-in-the-loop:**
3. `user_wants_alternatives_condition` - Nagro-route kapag sinabing "oo" ng user sa mga alternatibo
4. `user_declines_alternatives_condition` - Nagro-route kapag sinabing "hindi" ng user sa mga alternatibo


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Hakbang 5: Gumawa ng Decision Manager Executor

Ito ang **puso ng human-in-the-loop pattern**! Ang `DecisionManager` ay isang custom na `Executor` na:

1. **Tumatanggap ng feedback mula sa tao** sa pamamagitan ng mga `RequestResponse` na mga object
2. **Pinoproseso ang desisyon ng gumagamit** (oo/hindi)
3. **Nagtuturo ng workflow** sa pamamagitan ng pagpapadala ng mga mensahe sa mga angkop na ahente

Mga pangunahing tampok:
- Gumagamit ng `@handler` decorator para ipakita ang mga pamamaraan bilang mga hakbang ng workflow
- Tumatanggap ng `RequestResponse[HumanFeedbackRequest, str]` na naglalaman ng orihinal na request at sagot ng gumagamit
- Nagbibigay ng simpleng mensaheng "oo" o "hindi" na nagpapagana sa aming mga function sa kondisyon


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## Hakbang 6: Gumawa ng Pasadyang Display Executor

Parehong display executor mula sa conditional workflow - nagbibigay ng panghuling resulta bilang output ng workflow.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Hakbang 7: I-load ang mga Variable ng Kapaligiran

I-configure ang LLM client (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## Hakbang 8: Gumawa ng AI Agents at Executors

Gumagawa kami ng **anim na workflow components**:

**Mga Agents (nakabalot sa AgentExecutor):**
1. **availability_agent** - Tinitingnan ang availability ng hotel gamit ang tool
2. **confirmation_agent** - 🆕 Naghahanda ng kahilingan para sa kumpirmasyon ng tao
3. **alternative_agent** - Nagsusuggest ng alternatibong mga lungsod (kapag sumang-ayon ang user)
4. **booking_agent** - Nanghihikayat ng pag-book (kapag may available na mga kuwarto)
5. **cancellation_agent** - 🆕 Humahawak ng mensahe ng pagkansela (kapag tumanggi ang user)

**Espesyal na Executors:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` na nagpahinto sa workflow para sa input ng tao
7. **decision_manager** - 🆕 Custom executor na nagro-route base sa sagot ng tao (naka-define na kanina)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Hakbang 9: Buuhin ang Workflow kasama ang Tao-sa-Loop

Ngayon bubuuin natin ang workflow graph na may **conditional routing** kasama ang path na tao-sa-loop:

**Estruktura ng Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Pangunahing Mga Daan:**
- `availability_agent → confirmation_agent` (kapag walang kuwarto)
- `confirmation_agent → prepare_human_request` (pagbabago ng uri)
- `prepare_human_request → request_info_executor` (paghinto para sa tao)
- `request_info_executor → decision_manager` (palagi - nagbibigay ng RequestResponse)
- `decision_manager → alternative_agent` (kapag sinabi ng user na "oo")
- `decision_manager → cancellation_agent` (kapag sinabi ng user na "hindi")
- `availability_agent → booking_agent` (kapag may available na kuwarto)
- Lahat ng daan ay nagtatapos sa `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Hakbang 10: Patakbuhin ang Test Case 1 - Lungsod NA WALANG Availability (Paris na may Kumpirmasyon ng Tao)

Ipinapakita ng test na ito ang **buong human-in-the-loop na siklo**:

1. Humiling ng hotel sa Paris
2. sinuri ng availability_agent → Walang kuwarto
3. gumawa ang confirmation_agent ng tanong para sa tao
4. pinahihinto ng request_info_executor ang workflow at nagpapalabas ng `RequestInfoEvent`
5. **Nadidiskubre ng Application ang event at hinihikayat ang user sa console**
6. Nagta-type ang User ng "oo" o "hindi"
7. Ipinapadala ng Application ang tugon sa pamamagitan ng `send_responses_streaming()`
8. Iniruruta ng decision_manager base sa tugon
9. Ipinapakita ang panghuling resulta

**Pangunahing Pattern:**
- Gamitin ang `workflow.run_stream()` para sa unang pag-ulit
- Gamitin ang `workflow.send_responses_streaming(pending_responses)` para sa mga kasunod na pag-ulit
- Makinig sa `RequestInfoEvent` para matukoy kung kailangang may input mula sa tao
- Makinig sa `WorkflowOutputEvent` para makuha ang panghuling resulta


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Hakbang 11: Patakbuhin ang Test Case 2 - Lungsod NA MAY Availability (Stockholm - Hindi Kailangan ng Human Input)

Ipinapakita ng test na ito ang **direktang daan** kapag may mga available na kuwarto:

1. Humiling ng hotel sa Stockholm
2. sinusuri ng availability_agent → May mga kuwarto ✅
3. inirerekomenda ng booking_agent ang pag-book
4. ipinapakita ng display_result ang kumpirmasyon
5. **Hindi kailangan ng input mula sa tao!**

Nilalaktawan ng workflow ang human-in-the-loop na daan kapag may mga available na kuwarto.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Mga Pangunahing Punto at Pinakamahuhusay na Gawi sa Human-in-the-Loop

### ✅ Mga Natutunan Mo:

#### 1. **RequestInfoExecutor Pattern**
Ang human-in-the-loop pattern sa Microsoft Agent Framework ay gumagamit ng tatlong pangunahing sangkap:
- `RequestInfoExecutor` - Pinahihinto ang workflow at naglalabas ng mga kaganapan
- `RequestInfoMessage` - Base class para sa typed request payloads (i-subclass ito!)
- `RequestResponse` - Nag-uugnay ng mga tugon ng tao sa orihinal na mga kahilingan

**Mahalagang Pag-unawa:**
- Ang `RequestInfoExecutor` AY HINDI nangongolekta ng input mismo - pinahihinto lang nito ang workflow
- Kailangang makinig ang iyong application code para sa `RequestInfoEvent` at kolektahin ang input
- Dapat tawagin mo ang `send_responses_streaming()` na may dict na nagmamapa ng `request_id` sa sagot ng user

#### 2. **Streaming Execution Pattern**
```python
# Unang pag-ulit
stream = workflow.run_stream(initial_request)

# Mga sumunod na pag-ulit (pagkatapos ng input ng tao)
stream = workflow.send_responses_streaming(pending_responses)

# Laging iproseso ang mga kaganapan
events = [event async for event in stream]
```

#### 3. **Event-Driven Architecture**
Makinig sa mga partikular na kaganapan para kontrolin ang workflow:
- `RequestInfoEvent` - Kailangan ang input ng tao (hinarang ang workflow)
- `WorkflowOutputEvent` - Available na ang pangwakas na resulta (tapos na ang workflow)
- `WorkflowStatusEvent` - Mga pagbabago sa estado (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, atbp.)

#### 4. **Custom Executors with @handler**
Ipinapakita ng `DecisionManager` kung paano gumawa ng mga executor na:
- Gumamit ng `@handler` decorator para ipakita ang mga method bilang workflow steps
- Tumanggap ng typed messages (hal., `RequestResponse[HumanFeedbackRequest, str]`)
- Mag-route ng workflow sa pamamagitan ng pagpapadala ng mga mensahe sa ibang executor
- Ma-access ang context gamit ang `WorkflowContext`

#### 5. **Conditional Routing with Human Decisions**
Maaari kang gumawa ng mga condition function na sumusuri sa mga tugon ng tao:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Mga Real-World Applications:

1. **Approval Workflows**
   - Kumuha ng apruba ng manager bago iproseso ang expense reports
   - Kailangan ang human review bago magpadala ng mga automated na email
   - Kumpirmahin ang mga transaksyong may mataas na halaga bago ito isagawa

2. **Content Moderation**
   - I-flag ang mga kuwestiyonableng nilalaman para sa human review
   - Tanungin ang mga moderator na gumawa ng huling desisyon sa mga edge case
   - I-escalate sa mga tao kapag mababa ang kumpiyansa ng AI

3. **Customer Service**
   - Hayaan ang AI na hawakan ang mga rutinang tanong nang awtomatiko
   - I-escalate ang mga komplikadong isyu sa mga human agent
   - Tanungin ang customer kung gusto nilang makipag-usap sa isang tao

4. **Data Processing**
   - Hilingin sa mga tao na lutasin ang mga malabong data entries
   - Kumpirmahin ang mga interpretasyon ng AI sa mga hindi malinaw na dokumento
   - Pahintulutan ang mga user na pumili sa pagitan ng maraming wastong interpretasyon

5. **Safety-Critical Systems**
   - Kailangan ng kumpirmasyon ng tao bago ang mga hindi na mababaliktad na aksyon
   - Kumuha ng apruba bago ma-access ang sensitibong data
   - Kumpirmahin ang mga desisyon sa mga regulated na industriya (healthcare, finance)

6. **Interactive Agents**
   - Gumawa ng mga conversational bot na nagtatanong ng mga follow-up na tanong
   - Lumikha ng mga wizard na gumagabay sa mga user sa buong kumplikadong proseso
   - Disenyuhin ang mga agent na nakikipagtulungan sa mga tao nang hakbang-hakbang

### 🔄 Paghahambing: May Human-in-the-Loop kumpara Walang Human-in-the-Loop

| Katangian | Conditional Workflow | Human-in-the-Loop Workflow |
|---------|---------------------|---------------------------|
| **Execution** | Isang `workflow.run()` | Loop gamit ang `run_stream()` + `send_responses_streaming()` |
| **User Input** | Wala (fully automated) | Interactive na prompt gamit ang `input()` o UI |
| **Components** | Agents + Executors | + RequestInfoExecutor + DecisionManager |
| **Events** | AgentExecutorResponse lamang | RequestInfoEvent, WorkflowOutputEvent, atbp. |
| **Pausing** | Walang pag-pause | Humihinto ang workflow sa RequestInfoExecutor |
| **Human Control** | Walang kontrol ng tao | Gawa ng tao ang mga pangunahing desisyon |
| **Use Case** | Awtomasyon | Pakikipagtulungan at pagmamatyag |

### 🚀 Advanced Patterns:

#### Maramihang Punto ng Desisyon ng Tao
Maaari kang magkaroon ng maramihang `RequestInfoExecutor` node sa parehong workflow:
```python
.add_edge(agent1, request_info_1)  # Unang desisyon ng tao
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Pangalawang desisyon ng tao
.add_edge(decision_manager_2, final_agent)
```

#### Timeout Handling
Magpatupad ng mga timeout para sa mga tugon ng tao:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Default sa ligtas na opsyon
```

#### Malawak na Integrasyon ng UI
Sa halip na `input()`, isama sa web UI, Slack, Teams, atbp.:
```python
if isinstance(event, RequestInfoEvent):
    # Magpadala ng abiso sa paboritong channel ng gumagamit
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Conditional Human-in-the-Loop
Humingi lamang ng human input sa tiyak na mga sitwasyon:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Ipadala lamang sa tao kung mababa ang kumpiyansa o mataas ang halaga
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Pinakamahuhusay na Gawi:

1. **Laging I-subclass ang RequestInfoMessage**
   - Nagbibigay ng type safety at validation
   - Nagpapahintulot ng mas mayamang context para sa UI rendering
   - Nililinaw ang layunin ng bawat uri ng kahilingan

2. **Gumamit ng Makahulugang Prompt**
   - Isama ang context tungkol sa iyong tinatanong
   - Ipaliwanag ang mga kahihinatnan ng bawat pagpipilian
   - Panatilihing simple at malinaw ang mga tanong

3. **Harapin ang Hindi Inaasahang Input**
   - I-validate ang mga sagot ng user
   - Magbigay ng default para sa mga invalid na input
   - Magbigay ng malinaw na mga mensahe ng error

4. **Subaybayan ang Request ID**
   - Gamitin ang korelasyon sa pagitan ng request_id at mga tugon
   - Huwag subukang pamahalaan ang state nang manu-mano

5. **Disenyuhin para sa Non-Blocking**
   - Huwag ipahinto ang mga thread habang naghihintay ng input
   - Gumamit ng async na mga pattern sa buong proseso
   - Suportahan ang sabayang mga instance ng workflow

### 📚 Kaugnay na Mga Konsepto:

- **Agent Middleware** - Intercept ng mga tawag ng agent (nakaraang notebook)
- **Workflow State Management** - Panatilihin ang estado ng workflow sa pagitan ng mga run
- **Multi-Agent Collaboration** - Pagsamahin ang human-in-the-loop sa mga agent team
- **Event-Driven Architectures** - Gumawa ng mga reaktibong sistema gamit ang mga kaganapan

---

### 🎓 Binabati kita!

Nakamit mo na ang kahusayan sa human-in-the-loop workflows gamit ang Microsoft Agent Framework! Ngayon ay alam mo kung paano:
- ✅ Pahintuin ang workflows upang mangalap ng input mula sa tao
- ✅ Gamitin ang RequestInfoExecutor at RequestInfoMessage
- ✅ Pangasiwaan ang streaming execution gamit ang mga kaganapan
- ✅ Gumawa ng mga custom executor gamit ang @handler
- ✅ Mag-route ng workflows batay sa mga desisyon ng tao
- ✅ Magtayo ng interactive na AI agents na nakikipagtulungan sa mga tao

**Isang kritikal na pattern ito para sa pagbuo ng mapagkakatiwalaan, kontroladong mga AI system!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
